# Model Design and Decisions

The modeling layer estimates relative species use and environmental plausibility from engineered `h3`/`date` features. This section explains the main modeling choices and why they matter for interpreting the final risk products.

It is a guide to the model design, validation framing, and plausibility layer, not a model-training notebook.


## Modeling Inputs

The species-use model is trained from `data/modeling/species_training/`, which combines species telemetry support with the engineered environmental and static features. The active numerical feature set is defined in `src/riskscape/model/dataset.py` and includes:

- dynamic environmental features: SST, SSH, wind speed, log chlorophyll-a, anomalies, gradients, and seasonal encodings;
- static spatial features: bathymetry depth, slope, distance to coast, and encoded H3 centroid coordinates;
- species identity, encoded separately for the joint-species model.

The model-ready rows use `(h3, date, species)` as keys. Rows with telemetry support carry positive species use; rows for observed `species`/`date` combinations without telemetry in a given H3 cell are modeling zeros, not confirmed biological absences.


## Response Variable

Telemetry-derived species use is represented by a residence index:

$$
R(h,t,s)=C(h,t,s)\times I(h,t,s)
$$

where $C$ is telemetry record count and $I$ is tracked individual count for H3 cell $h$, date $t$, and species $s$. The modeling target is log transformed:

$$
Y(h,t,s)=\log(1+R(h,t,s))
$$

The log transform keeps the model focused on relative intensity while reducing the influence of very large residence-index values.


## Joint-Species Design

The active production path uses a joint-species model. Species identity is encoded as categorical indicators and appended to the numerical feature matrix. This lets one model learn shared environmental structure while still allowing species-specific responses.

This design is useful for the Falkland case study because BBAL and SAFS share the same environmental feature grid and prediction domain, but the telemetry support is uneven across space and time.


## Learner Selection

The workflow evaluated several learners during screening, including histogram gradient boosting, random forest, Extra Trees, and GMM/Bayesian-GMM candidates. Extra Trees was selected as the primary species-use learner because it gave strong predictive performance while remaining robust to nonlinear feature interactions and mixed static/dynamic predictors.

The active species-use model artifact is:

```text
data/
└── modeling/
    └── models/
        └── extra_trees_som_hierarchical_k30_5fold_blockcv/
            └── species_model_joint.joblib
```

The production model is trained on balanced rows after the validation design has been selected. Training remains available through the model scripts, but the public workflow can also use the archived model products directly.


## Validation Design

Row-random validation is treated as an initial benchmark only. Randomly mixed `h3`/`date` rows can overstate transferability because nearby cells, nearby dates, and similar environmental states may appear in both train and test partitions.

The selected validation framing uses SOM-hierarchical k=30 seascape classes as environmental groups in five-fold grouped cross-validation. Those group labels are read from `data/modeling/environmental_regimes/`, the consolidated `h3`/`date` table that stores the selected seascape labels and Bayesian GMM environmental components.

The retained validation summary is:

```text
data/
└── modeling/
    └── metrics/
        └── species_model_block_cv_selected_som_k30_5fold.csv
```


## Balancing and Weights

Zero-use rows greatly outnumber positive-use rows. For model fitting, all positive rows are retained and an equal number of zero-use rows is sampled. This prevents the model from being dominated by zeros while keeping the contrast between used and unused `h3`/`date`/`species` rows.

Higher-use observations receive larger sample weights:

$$
w(h,t,s)=1+R(h,t,s)^{0.75}
$$

This weighting keeps high-support telemetry events influential without letting the largest values completely dominate the fit.


## Plausibility Layer

The Bayesian GMM model is retained as an environmental-support layer rather than as the primary species-use learner. It estimates how similar each `h3`/`date`/`species` feature vector is to the environmental conditions associated with observed positive telemetry use.

The active plausibility model artifact is:

```text
data/
└── modeling/
    └── models/
        └── bayesian_gmm_k30/
            └── species_model_joint.joblib
```

The resulting plausibility score is an environmental-support diagnostic. It should not be read as a species-presence probability.


## Hybrid Prediction Decision

The active prediction mode combines the Extra Trees species-use model with the Bayesian GMM plausibility layer using a presence gate. The gate reduces species-use intensity where environmental plausibility is low while preserving the Extra Trees prediction as the main species-use estimate.

The gate is species-specific:

$$
g(s,h,t)=1-c_s(1-P(s,h,t))
$$

where $P$ is plausibility and $c_s$ is the maximum allowed cut for species $s$. The current configuration uses a conservative maximum cut of 0.1 for both BBAL and SAFS.

The gated species-use prediction is:

$$
U_{hybrid}=\log\left(1+(\exp(U_{ML})-1)g\right)
$$

Conceptually, the expression does three things:

- $U_{ML}$ is on the log scale, so $\exp(U_{ML})-1$ converts it back to the residence-index scale.
- $g$ is the plausibility gate. When plausibility is high, $g$ stays close to 1 and the Extra Trees prediction is mostly preserved. When plausibility is low, $g$ becomes smaller and reduces the predicted species-use intensity.
- The outer $\log(1 + \ldots)$ returns the gated value to the same log scale used by the model outputs.

This keeps plausibility as a constraint on environmental support, not as a replacement for the species-use model. The gate can dampen a prediction made under unfamiliar environmental conditions, but it does not create a new species-presence probability.


## Terminology for Prediction Products

The model outputs are easier to interpret if the layers stay separate:

- **Species use** is the model's relative prediction of telemetry-derived residence intensity.
- **Plausibility** is environmental support from the Bayesian GMM layer; it asks whether a prediction is being made under familiar environmental conditions.
- **Hybrid species use** is the Extra Trees species-use prediction after the plausibility gate is applied.
- **Risk products** are built downstream, when species-use predictions are combined with fishing exposure or operational summaries.

Keeping these terms separate prevents plausibility from being mistaken for presence probability and prevents relative risk from being read as absolute bycatch probability.


## Retained Validation Metrics

The selected validation summary is small enough to inspect directly. The preview below reads the retained SOM k30 five-fold grouped cross-validation metrics from the archived modeling products.


In [4]:
from pathlib import Path
import pandas as pd
import yaml

config = yaml.safe_load(Path("../config.yaml").read_text())
metrics_path = (
    Path("..")
    / config["paths"]["data"]
    / "modeling"
    / "metrics"
    / "species_model_block_cv_selected_som_k30_5fold.csv"
)

row = pd.read_csv(metrics_path).iloc[0]
summary = pd.DataFrame(
    {
        "metric": [
            "validation variant",
            "model",
            "model type",
            "species",
            "CV folds",
            "R2 mean",
            "RMSE mean",
            "MAE mean",
            "log R2 mean",
            "log RMSE mean",
            "log MAE mean",
        ],
        "value": [
            row["validation_variant"],
            row["model"],
            row["model_type"],
            row["species"],
            int(row["cv_folds"]),
            round(row["r2_mean"], 3),
            round(row["rmse_mean"], 3),
            round(row["mae_mean"], 3),
            round(row["r2_log_mean"], 3),
            round(row["rmse_log_mean"], 3),
            round(row["mae_log_mean"], 3),
        ],
    }
)
summary


,metric,value
0,validation variant,som_k30_5fold
1,model,extra_trees
2,model type,joint_species
3,species,all
4,CV folds,5
5,R2 mean,0.769
6,RMSE mean,52.453
7,MAE mean,3.32
8,log R2 mean,0.518
9,log RMSE mean,0.538


In [5]:
from pathlib import Path
import yaml

config = yaml.safe_load(Path("../config.yaml").read_text())
data_dir = Path("..") / config["paths"]["data"]

artifacts = [
    data_dir / "modeling" / "models" / "extra_trees_som_hierarchical_k30_5fold_blockcv" / "species_model_joint.joblib",
    data_dir / "modeling" / "models" / "bayesian_gmm_k30" / "species_model_joint.joblib",
    data_dir / "modeling" / "metrics" / "species_model_block_cv_selected_som_k30_5fold.csv",
    data_dir / "modeling" / "metrics" / "species_model_extra_trees_som_hierarchical_k30_5fold_blockcv_production_metrics.csv",
]

for artifact in artifacts:
    print(f"{artifact.relative_to(Path('..'))}: {'present' if artifact.exists() else 'missing'}")


data/modeling/models/extra_trees_som_hierarchical_k30_5fold_blockcv/species_model_joint.joblib: present
data/modeling/models/bayesian_gmm_k30/species_model_joint.joblib: present
data/modeling/metrics/species_model_block_cv_selected_som_k30_5fold.csv: present
data/modeling/metrics/species_model_extra_trees_som_hierarchical_k30_5fold_blockcv_production_metrics.csv: present
